# **Traditional RAG Pipeline**

### **1: Data Ingestion**

#### Step1: Instal Packages and Import Required Libraries

In [1]:
# Import required libraries
import os # To check and create file path 
import numpy as np # Use for handle numpy arrays
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader # PDF document loader from Langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter # ***
from pathlib import Path # ***
from sentence_transformers import SentenceTransformer # Embedding model library
import chromadb # Vector Database
from chromadb.config import Settings # ***
from sklearn.metrics.pairwise import cosine_similarity # Search for similarity to retreival process
import uuid # Give ID for every record we send to VectorDB
from typing import List, Dict, Any, Tuple # ***

/workspaces/Retrieval-Augmented-Generation-RAG/Traditional_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step2: Read and Load files from directory
For this practice we are going to use pdf files and pdf loader from langchain. It can be changed by any other files type loaders.

In [2]:
# Function to read and load pdf files
def process_pdf_files(directory): #  Use "driectory" variable as input to indicate files directory path
    """ Process all PDF files in a directory"""

    all_documents = [] # Create an empty list to put all docs in it
    pdf_dir = Path(directory) # Put path of directory in a variable

    # Find all pdf files recursivaly
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process") # Print the length of pdf_files variable to know how many files we found to process

    # File loading process
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}") # print a message to start of loading process

        # Use pdf loder to process files
        try:
            loader = PyPDFLoader(str(pdf_file)) # Use PyPDFLoader for lightweight applications, simple text extraction
            # loader - PyMuPDFLoader(str(pdf_file)) # Use PyMuPDFLoader for large documents, and advanced feature extraction (tables, images, annotations)
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source-file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents) # appends all documnets from for loop to the end of the list
            print(f"Loaded {len(documents)} pages.")

        except Exception as e:
            print(f"Error: {e}") # print error message
    
    print(f"\n Total documents loaded: {len(all_documents)}") # Print number of loaded documnets
    return all_documents

# Call fuction to process all pdf files in the directory
all_pdf_documents = process_pdf_files("../data/pdf_files")

Found 8 PDF files to process

 Processing: Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf
Loaded 6 pages.

 Processing: AVERE - Improving Audiovisual Emotion Reasoning with Preference Optimization.pdf
Loaded 11 pages.

 Processing: Emotion-LLaMAv2 and MMEVerse - A New Framework and Benchmark for Multimodal Emotion Understanding.pdf
Loaded 15 pages.

 Processing: Multimodal Classification Algorithms for Emotional Stress Analysis with an ECG-Centered Framework.pdf
Loaded 31 pages.

 Processing: XEmoGPT - An Explainable Multimodal Emotion Recognition Framework.pdf
Loaded 11 pages.

 Processing: HybridSense-LLM - A Structured Multimodal Framework for LLM Based Wellness Prediction.pdf
Loaded 19 pages.

 Processing: Improving cognitive stress classification via multimodal EEG and ECG fusion.pdf
Loaded 20 pages.

 Processing: EmoWork-A Multimodal Dataset for Assessing Emotion, Stress, and Emotional Workload.pdf
Loaded 14 pages.

 Total documents loa

#### Step3: Split documnets texts and creat chunks (Chunking)
We are going to use all documnets we created in previuse step and split the to chunks

In [3]:
# Function to split text and chunking
def split_documents_to_text(documents, chunk_size=1000, chunk_overlap=200): # A function with 3 variables
    """ The function uses split technique to dvide documents into smaller text fpr better RAG performance.

    It Uses 3 variable:
    1) "documets" from outsde of the functtion as input data,
    2) "chunck_size" and "chunk_overlap" to define the size of each chunk and how many characters it allowed to have in common."""
    
    # Configuration of text spiter using "RecursiveCharacterTextSplitter" library
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    # Perform text spliter on documents
    split_docs = text_spliter.split_documents(documents) # Split documents and store in split_docs variable
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks") # message to number of created chunks out of number of documents

    # Show sample of chunks to check if it works correctly
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:300]}...") # Show the first 200 charactors of first document page content
        print(f"Metadata: {split_docs[0].metadata}") # Show the metadata of first document

    return split_docs

chunks = split_documents_to_text(all_pdf_documents) # Implement the function by sending loaded documents in "all_pdf_documents"
chunks[:5]

Split 127 documents into 714 chunks

Example chunk:
Content: Integrating Fine-Grained Audio-Visual Evidence for
Robust Multimodal Emotion Reasoning
Zhixian Zhao †‡, Wenjie Tian †‡ and Lei Xie ∗‡,Senior Member , IEEE
‡Audio, Speech and Language Processing Group (ASLP@NPU),
Northwestern Polytechnical University, Xi’an, China
Abstract—Multimodal emotion analysis...
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260505191336', 'moddate': '2026-05-05T19:47:53+02:00', 'source': '../data/pdf_files/Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source-file': 'Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260505191336', 'moddate': '2026-05-05T19:47:53+02:00', 'source': '../data/pdf_files/Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source-file': 'Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf', 'file_type': 'pdf'}, page_content='Integrating Fine-Grained Audio-Visual Evidence for\nRobust Multimodal Emotion Reasoning\nZhixian Zhao †‡, Wenjie Tian †‡ and Lei Xie ∗‡,Senior Member , IEEE\n‡Audio, Speech and Language Processing Group (ASLP@NPU),\nNorthwestern Polytechnical University, Xi’an, China\nAbstract—Multimodal emotion analysis is shifting from static\nclassification to generative reasoning. Beyond simple label pre-\ndiction, robust affective reasoning must synthesize fine-grained\nsignals such as facial micro-expressions and prosodic which shifts\nto decode the la

#### Step4: Implement Embedding
We are going to use pre trained LLMs for embedding; convert text to vectors.

In [4]:
# Create a class to perform embedding
class EmbeddingManager:
    """ Generate documnets embedding using SentenceTransformer from Huggingface."""

    # Class initialize function
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # Implement function to load model

    # Function to load model (a protective function)
    def _load_model(self):
        """ Load the SentenceTransformer model"""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name) # Implement Model
            print(f"\nModel Loaded Successfully.")
            print(f"Embedding Dimention: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise

    # Function to generate embedding
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        generation embedding for list of texts

        Args: List of text string to convert to embedding vector

        Return: numpy array of embedding vector with shape (len(text), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded.")
        
        # Perform embedding
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

# Initialize the Embedding Manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:02<00:00, 50.91it/s]



Model Loaded Successfully.
Embedding Dimention: 384


#### Step5: VectoreStore
In this step we are going to prepare the vector database and store all vector embedding into a VectorDB.

Warning (Bug report): This pipeline designed to use spesific documents and store embeddings once for all. If there is a need to start over the "vector_store" folder must be deleted and vector store process should start over.

In [5]:
# Define a class to handle VectorStore process
class VectorStore:
    """ Manage embedding vectors int ChromaDB vector store"""

    # Function to initialize VectoreStore class
    def __init__(self, collection_name: str = ("pdf_documents"), persist_directory: str = "../vector_store"):
        """ Initialize the vector store

        Args:
            1) collection_name: name of the ChromaDB collection,
            2) persist_directory: directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() # Function to initialize vector store

    # Function to initialize client and collection
    def _initialize_store(self):
        """ Initialize ChromaDB client and collection """

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True) # Check excitance of the directory path, if not exsit it creates one with the name
            self.client = chromadb.PersistentClient(path = self.persist_directory) #  Creates a persistent ChromaDB client: a connection to a local vector database that saves data to disk

            # Retrieve an existing collection by name, or create a new one if it doesn't exist
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name, # Name used to identify this collection
                metadata = {"Description": "vector embeddings of pdf documnets to use in RAG"} # Optional metadata: describing the collection's purpose
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Exsiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    # Function to add documents to vector store
    def add_documents(self, documents: List[Any], embeddings: np.ndarray): # Method accepts a list of LangChain documents and their numpy embeddings
        """ Add documents and their corresponding embeddings to the vector store.

            Args:
                1) documents: list of documents created and loaded by Langchain document loader
                2) embeddings: corresponding for the documents
        """

        # Check if number of documents and thier corresponding embedding match
        if len(documents) != len(embeddings): # Validates that every document has exactly one embedding
            raise ValueError("Number of documents must match with number of embeddings") # Aborts early with a clear error if counts differ
        
        print(f"Adding {len(documents)} documents to vector store ...") # Logs how many documents are about to be inserted

        # Prepare data for ChromaDB
        ids = [] # Will hold a unique string ID for each document
        metadatas = [] # Will hold a metadata dict for each document
        documents_text = [] # Will hold the raw text content of each document
        embeddings_list = [] # Will hold each embedding converted to a plain Python list

        # Create a loop to prepare data
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): # Iterates over paired (document, embedding) tuples with an index
            # Generate a unique ID using 8 hex chars from a UUID plus the loop index to avoid collisions
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" 
            ids.append(doc_id) # Adds the generated ID to the ids list

            # Prepare metadata
            metadata = dict(doc.metadata) # Copies the document's existing metadata dict so the original is not mutated
            metadata["doc_index"] = i # Stores the document's position in the batch for later reference
            metadata["content_length"] = len(doc.page_content) # Records character count of the document text as a metadata field
            metadatas.append(metadata) # Adds the enriched metadata dict to the metadatas list

            # Prepare documents contetnt
            documents_text.append(doc.page_content) # Extracts the raw text string from the LangChain document and adds it to the list

            # Prepare documents corresponding embeddings
            embeddings_list.append(embedding.tolist()) # Converts the numpy array embedding to a plain Python list, as required by ChromaDB

        # Add to collection
        try:
            # Calls ChromaDB's add method to insert all prepared data in one batch
            self.collection.add(
                ids = ids, # Unique identifiers for each entry
                metadatas = metadatas, # Metadata dicts enriched with index and content length
                documents = documents_text, # Raw text content for each document
                embeddings = embeddings_list # Vector embeddings as plain Python lists
            )
            print(f"Successfully added {len(documents)} documents to vector store ")
            print(f"Total documents in collection: {self.collection.count()}") # Logs the new total count in the ChromaDB collection

        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

# Instantiates the VectorStore class, initializing the ChromaDB client and collection
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Exsiting documents in collection: 0


#### Step6: Convert text to embeddings
In this step we are going to use Embeddingmanager and VectorStore classes we created to implement text to embeddings convertion and store in vector database.

In [6]:
# Extract the raw text content from each document chunk into a list
texts = [doc.page_content for doc in chunks]
texts[:5]

# Generate vector embeddings for each text using the embedding model
embeddings = embedding_manager.generate_embeddings(texts)

# Store the document chunks alongside their embeddings in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 714 texts...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches: 100%|██████████| 23/23 [00:50<00:00,  2.19s/it]


Generated embeddings with shape: (714, 384)
Adding 714 documents to vector store ...
Successfully added 714 documents to vector store 
Total documents in collection: 714


### **2: User Query Process and Retrieval Pipeline**

In [7]:
# RAG Retriever Class
class RAGRetriever:
    """Handle query base retrieval from the vector store"""

    # Function to initialize process
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """ Inistilaize the retriever

            Args:
                1) vector_store: Vector store containing documents embeddings
                2) embedding_manager: manage to generating query embedding
        """

        self.vector_store = vector_store # Store reference to the vector database (holds document embeddings)
        self.embedding_manager = embedding_manager # Store reference to the embedding manager (converts text to vectors)

    # Function to retrieve context based on query
    def retrieve(self, query: str, top_k: int = 5, score_threshold = 0.0) -> List[Dict[str, Any]]:
        """ retrieve relevant documents for spesific query

            Args:
               1) query: the search query
               2) top_k: number of top results to return
               3) score_threshold: minimum similarity score threshold
        """
        print(f"Retrieving documents for query '{query}'") # Log the incoming query
        print(f"\nTop_k {top_k}, Score_Threshold {score_threshold}") # Log retrieval parameters

        # Generate query embedding - Convert the query string into a vector 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0] # [0] unpacks the first (only) result

        try:
            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()], # Pass query vector as a list (ChromaDB expects a list of embeddings)
                n_results = top_k # Limit results to top_k closest matches
            )

            # Create empty list to hold filtered, formatted results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]: # Check that ChromaDB returned non-empty results
                documents = results['documents'][0] # Extract the raw document texts (inner [0] = first query's results)
                metadatas = results['metadatas'][0] # Extract metadata dicts paired to each document
                distances = results['distances'][0] # Extract similarity distances (lower = more similar in ChromaDB)
                ids = results['ids'][0] # Extract unique document IDs

                # Similarity Calculation Process: Iterate over all result fields together, with index i for ranking
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similatiry score (ChromaDB uses Cosine distance)
                    similarity_score = 1 - distance # Invert cosine distance to get similarity (1.0 = identical, 0.0 = unrelated)

                    # Skip documents that don't meet the minimum similarity requirement
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,           # Unique identifier of the matched document
                            'content': document,    # The actual text content of the document
                            'metadata': metadata,   # Associated metadata (e.g. source, page number)
                            'similarity_score': similarity_score, # How similar the document is to the query (0–1)
                            'distance': distance,   # Raw cosine distance returned by ChromaDB
                            'rank': i + 1           # 1-based rank position in the result list
                        })                
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)") # Log how many docs passed the threshold
            else:
                print("No documents found") # Log when ChromaDB returns an empty result set

            return retrieved_docs # Return the final filtered and formatted list of documents
        
        except Exception as e:
            print(f"Error during the retrieval: {e}") # Log any unexpected errors without crashing the program
            return[] # Return an empty list as a safe fallback on failure

# Instantiate the retriever with the shared vector store and embedding manager
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [8]:
# Simple question to test the retrive process
rag_retriever.retrieve("What is Multimodal dataset?")

Retrieving documents for query 'What is Multimodal dataset?'

Top_k 5, Score_Threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.30it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a0b11622_132',
  'content': 'tion 6 concludes the paper and outlines future directions.\n2 RELATEDWORK\nTo contextualize our contributions, this section reviews\nthree relevant research areas: (1) multimodal emotion un-\nderstanding, (2) multimodal large language models, and\n(3) instruction tuning for cross-modal learning.\n2.1 Multimodal Emotion Understanding\nMultimodal emotion understanding aims to integrate vi-\nsual, auditory, and textual cues to identify human affective\nstates (emotion recognition) and to infer their underlying\ncauses (emotion reasoning) [33], [34]. Early research in this\narea primarily adopted discriminative modeling paradigms\nunder controlled laboratory settings, as exemplified by\ndatasets such as IEMOCAP [27]. Subsequent efforts ex-\ntended these approaches to more diverse and unconstrained\nenvironments, including movies, television series, and in-\nthe-wild scenarios [5], [28], [29], [35]. Conventional method-\nologies typically extracted 

In [9]:
rag_retriever.retrieve("What is stress detection?")

Retrieving documents for query 'What is stress detection?'

Top_k 5, Score_Threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4d0cab2e_370',
  'content': 'By consolidating current knowledge and identifying critical methodological gaps, this\nreview provides a structured foundation for developing reliable, interpretable, and scal-\nable stress recognition systems that can move beyond laboratory settings toward real-\nworld deployment.\nAuthor Contributions: Conceptualization, X.Z. and M.X.; methodology, X.Z.; formal analysis, X.Z.;\ninvestigation, X.Z.; resources, M.X.; data curation, X.Z.; writing—original draft preparation, X.Z.; writ-\ning—review and editing, X.Z. and H.Z.; visualization, X.Z.; supervision, M.X.; project administration,\nM.X. All authors have read and agreed to the published version of the manuscript.\nFunding: This research received no external funding.\nInstitutional Review Board Statement: Not applicable.\nInformed Consent Statement: Not applicable.\nData Availability Statement: No new data were created or analyzed in this study.\nConflicts of Interest: The authors declare n

### **3: Query Retrival Pipeline**

#### Integrating VectorStore Pipeline Context With LLM

In [ ]:
# Simple RAG pipeline with Groq LLM - import required libraries
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

# # Use Groq API to initialize Groq LLM (Set your GROQ_API_KEY in environment)
groq_api_key = "***"

llm = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# Basic RAG Function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve context
    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context found to answer the question"
    
    # Generate the answer
    prompt=f"""Use the following context to answer the question consicely
            Context: 
            {context}

            Question: {query}

            Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [13]:
answer = rag_simple("What is moltimodal stress detection?", rag_retriever, llm)
print(answer)

Retrieving documents for query 'What is moltimodal stress detection?'

Top_k 3, Score_Threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Multimodal stress detection refers to the integration of various data sources, including both subjective psychometric data (e.g., self-reported stress levels) and physiological measurements (e.g., heart rate, skin conductance), to more accurately capture emotional stress.


#### Enhanced RAG Retrival Pipeline Features

In [14]:
# RAG Functuin to retrive more information and features using retrival pipeline
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """
    RAG retrieval pipeline with extra features

    Retrun answer, sources, confidence score, and optional full context.
    """

    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'source': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question consicely. \nContext:{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context = context, query = query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Emotion recognition?", rag_retriever, llm, top_k = 3, min_score = 0.1, return_context = True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query 'What is Emotion recognition?'

Top_k 3, Score_Threshold 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Emotion recognition is the ability of artificial agents or models to perceive and understand users' emotional states, including recognizing emotional states and reasoning about their underlying causes and contextual expressions.
Sources: [{'source': '../data/pdf_files/Emotion-LLaMAv2 and MMEVerse - A New Framework and Benchmark for Multimodal Emotion Understanding.pdf', 'page': 0, 'score': 0.22650021314620972, 'preview': 'action [1]–[4]. Accurate emotion understanding enables ar-\ntificial agents to perceive and adapt to users’ affective states,\nfostering empathy and effective communication in domains\nsuch as education, healthcare, robotics, and counseling. In\ncomputational settings, emotion understanding extends be-\n...'}, {'source': '../data/pdf_files/Emotion-LLaMAv2 and MMEVerse - A New Framework and Benchmark for Multimodal Emotion Understanding.pdf', 'page': 14, 'score': 0.21134120225906372, 'preview': 'international conference on Multimedia, 2016, pp. 1365–1374.\n[2] 

#### Advanced RAG Retrival Pipeline Features

In [15]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k = top_k, score_threshold = min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end = '', flush = True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context = context, question = question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Emotion recognition?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query 'What is Emotion recognition?'

Top_k 3, Score_Threshold 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
action [1]–[4]. Accurate emotion understanding enables ar-
tificial agents to perceive and adapt to users’ affective states,
fostering empathy and effective communication in domains
such as education, healthcare, robotics, and counseling. In
computati

onal settings, emotion understanding extends be-
yond categorical prediction and requires models to jointly
recognize emotional states and reason about their underly-
ing causes and contextual expressions. Unlike conventional
recognition tasks, emotion understanding integrates percep-
Corresponding author (zhiqics@uw.edu).
•This work was primarily conducted by the teams at Shenzhen T echnology
University and the University of Washington.
•Xiaojiang Peng, Jingyi Chen, Zebang Cheng, and Yuxiang Lin are with
Shenzhen T echnology University. Bao Peng is with Shenzhen University
of Information T echnology.
•Fengyi Wu, Yifei Dong, Qiyu Hu, Shuyuan T u, Huiting Huang, and
Zhi-Qi Cheng are with the University of Washington.

international conference on Multimedia, 2016, pp. 1365–1374.
[2] I. S. MacKenzie,Human-computer interaction: An empirical research
perspective. Elsevier, 2024.
[3] M. Imani and G. A. Montazer, “A survey of emotion recognition
methods with emphasis on e-learning environment